<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Un hash SHA-256 y un estado de puerta; `champion` solo si la puerta pasa.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: CI completo con varios checks, escenarios adicionales y seguimiento externo.
<!-- OPS:MAT-02:fin -->

# L18 · CI local: software rápido, estadística lenta

**S18 · Lectura: Cap. 20 · 40–55 min**

## La pregunta

¿Qué debe bloquear una puerta antes de publicar?

## Qué harás

Combinar tests de software con checks estadísticos locales.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una puerta que acepta un artefacto válido y rechaza uno incompleto.

> **Idea clave:** Que el código funcione no demuestra que el efecto siga siendo defendible.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, tempfile, shutil
from pathlib import Path
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
N_PERM = R(10, 40)
sc = load_scenario("synthetic/inference/ihdp_medium")
REF = sc.train.copy(); COVS = list(sc.covariates); T_, Y_ = sc.treatment, sc.outcome
HASH_REF = sha256_df(REF[COVS + [T_, Y_]])
print("referencia congelada:", len(REF), "filas · hash:", HASH_REF[:30], "…")

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Tres criterios de diseño (§20.3)
1. **Referencia congelada y versionada:** sin datos fijos, una prueba estadística no distingue una regresión del código de un cambio en los datos. → guardamos su **hash** y lo comprobamos.
2. **Umbrales fijados antes:** relajar el umbral cuando falla es justo lo que el pipeline existe para evitar.
3. **Un fallo estadístico lo decide una persona:** no se «arregla el código hasta que pase»; puede ser un cambio genuino del mundo.

Los umbrales se escriben **antes de ejecutar nada** y no se tocan:

In [ ]:
UMBRALES = {"placebo_cobertura_min": 0.85,     # fracción de placebos cuyo IC contiene 0
            "smd_ponderado_max": 0.10,           # balance de TODAS las covariables tras ponderar
            "efecto_z_max": 3.0}                 # |θ − θ_ref| / sqrt(se² + se_ref²)
print(UMBRALES)

## 2. El modelo bajo prueba
`ajustar(df, covs)` es el pipeline de A2 reducido (DML-PLR con cross-fitting K = 3 y ExtraTrees pequeños, para que la etapa lenta sea corta). Fija la **estimación de referencia** con el modelo sano.

In [ ]:
fast = lambda: ExtraTreesRegressor(n_estimators=20, min_samples_leaf=8, max_features=0.8, random_state=0)
def ajustar(df, covs, K=3, tratamiento=None):
    X = df[covs].to_numpy(float); y = df[Y_].to_numpy(float); t = df[T_].to_numpy(float) if tratamiento is None else np.asarray(tratamiento, float)
    n = len(y); ry = np.zeros(n); rd = np.zeros(n)
    for te in kfold(n, K, SEED):
        tr = np.setdiff1d(np.arange(n), te)
        ry[te] = y[te] - fast().fit(X[tr], y[tr]).predict(X[te]); rd[te] = t[te] - fast().fit(X[tr], t[tr]).predict(X[te])
    th = (rd * ry).sum() / (rd * rd).sum(); psi = (ry - th * rd) * rd / np.mean(rd ** 2)
    return float(th), float(np.sqrt(np.mean(psi ** 2) / n))
TH_REF, SE_REF = ajustar(REF, COVS)
print("referencia (modelo sano):", fmt_ci(TH_REF, SE_REF))

## 3. Etapa 1 — software (rápida)
Lo que haría `pytest` y `compileall`: ¿importa?, ¿devuelve números finitos?, ¿es reproducible?

In [ ]:
def sw_compila(): compile("def f(x):\n    return x", "<src>", "exec"); return True
def sw_finito(): th, se = ajustar(REF, COVS); return bool(np.isfinite(th) and se > 0)
def sw_reproducible(): return ajustar(REF, COVS) == ajustar(REF, COVS)
ETAPA_SW = {"compila": sw_compila, "devuelve números finitos": sw_finito, "es reproducible (misma semilla)": sw_reproducible}
print({k: f() for k, f in ETAPA_SW.items()})

## 4. Por qué el Listado 20.2 no prueba nada (E3)
El listado calcula `D_placebo = rng.permutation(...)` y **nunca lo usa**; después comprueba que el IC del efecto del modelo *ya ajustado* contenga 0. Replicamos esa lógica:

In [ ]:
def test_placebo_del_libro(theta, se, semilla):
    D_placebo = np.random.default_rng(semilla).permutation(REF[T_].to_numpy())      # se calcula…
    return (theta - 1.96 * se) <= 0 <= (theta + 1.96 * se)                          # …y no se usa
print("¿el «placebo» del libro cambia con la semilla?", {s: test_placebo_del_libro(TH_REF, SE_REF, s) for s in (0, 1, 2)})
th0, se0 = ajustar(REF.assign(**{Y_: np.random.default_rng(1).normal(size=len(REF))}), COVS)        # mundo SIN efecto
print("modelo sano (efecto real) →", test_placebo_del_libro(TH_REF, SE_REF, 0), "· modelo en un mundo sin efecto →", test_placebo_del_libro(th0, se0, 0))

**Mira el resultado.** ¿Qué mide en realidad ese test, y por qué bloquearía un modelo sano?

**Respuesta:**

Mide si el IC del **efecto real** contiene 0, no si el pipeline inventa efectos. Con un efecto verdadero grande (como el de IHDP) el test **falla siempre** y bloquea un modelo sano; en un mundo sin efecto pasa aunque el pipeline esté roto. Y no depende de la permutación: da lo mismo con cualquier semilla. Un placebo exige **reajustar el modelo con el tratamiento permutado**.



## 5. Completa tú: el placebo correcto
`test_placebo(df, covs, n_perm, umbral)`: para cada una de `n_perm` permutaciones del tratamiento, **reajusta** con `ajustar(df, covs, tratamiento=t_perm)` y comprueba si el IC 95 % contiene 0. Devuelve `True` si la fracción de placebos con IC que contiene 0 es ≥ `umbral`. Usa una semilla fija (`np.random.default_rng(0)`), por reproducibilidad.

In [ ]:
# ✏️ Completa tú — test_placebo
# Pista: Dentro del bucle: t_perm = r.permutation(t); th, se = ajustar(df, covs, tratamiento=t_perm).
def test_placebo(df, covs, n_perm=N_PERM, umbral=UMBRALES["placebo_cobertura_min"]):
    """True si, tras permutar el tratamiento y REAJUSTAR, la fracción de ICs que contienen 0 es ≥ umbral."""
    r = np.random.default_rng(0); t = df[T_].to_numpy(float)
    raise NotImplementedError("completa test_placebo")

In [ ]:
print("placebo del modelo sano:", test_placebo(REF, COVS))

## 6. Etapa 2 — estadística (lenta)
Además del placebo: (a) la **referencia sigue congelada** (hash), (b) **balance ponderado** de todas las covariables y (c) el **efecto frente a la referencia**.

In [ ]:
def smd_pond(x, t, w):
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    v1, v0 = np.average((x[t == 1] - m1) ** 2, weights=w[t == 1]), np.average((x[t == 0] - m0) ** 2, weights=w[t == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)
def test_balance(df, covs, todas=COVS, umbral=UMBRALES["smd_ponderado_max"]):
    t = df[T_].to_numpy(float); Xc = df[covs].to_numpy(float)
    e = np.clip(LogisticRegression(max_iter=500).fit(Xc, t).predict_proba(Xc)[:, 1], .02, .98); w = t / e + (1 - t) / (1 - e)
    return bool(max(abs(smd_pond(df[c].to_numpy(float), t, w)) for c in todas) <= umbral)
def test_efecto(df, covs, umbral=UMBRALES["efecto_z_max"]):
    th, se = ajustar(df, covs); return bool(abs(th - TH_REF) / np.sqrt(se ** 2 + SE_REF ** 2) <= umbral)
def test_referencia(df): return sha256_df(df[COVS + [T_, Y_]]) == HASH_REF
def etapa_estadistica(df, covs):
    return {"referencia congelada (hash)": lambda: test_referencia(df), "placebo con refit": lambda: test_placebo(df, covs),
            "balance ponderado de todas las covariables": lambda: test_balance(df, covs), "efecto frente a la referencia": lambda: test_efecto(df, covs)}
sano = {k: f() for k, f in etapa_estadistica(REF, COVS).items()}
print(sano)

## 7. Sabotaje: quitar un control clave (E4)
El ejercicio del libro dice que quitar una covariable de control «rompe el placebo». Probamos: quitamos del modelo la covariable que más pesa sobre tratamiento y resultado (un confusor importante) y ejecutamos las cuatro pruebas.

In [ ]:
# el «control clave» = la covariable que más pesa a la vez sobre el tratamiento y el resultado
CLAVE = max(COVS, key=lambda c: abs(np.corrcoef(REF[c], REF[T_])[0, 1]) * abs(np.corrcoef(REF[c], REF[Y_])[0, 1]))
COVS_MAL = [c for c in COVS if c != CLAVE]
print("control clave que se quita:", CLAVE)
mal = {k: f() for k, f in etapa_estadistica(REF, COVS_MAL).items()}
th_m, se_m = ajustar(REF, COVS_MAL)
show(pd.DataFrame({"modelo sano": sano, f"sin {CLAVE} (sabotaje)": mal}).replace({True: "pasa", False: "FALLA"}))
print(f"efecto sano {TH_REF:.3f} → sin {CLAVE} {th_m:.3f}")

**Mira el resultado.** ¿Qué prueba NO detecta el sabotaje y cuáles sí? ¿Qué corrige esto del enunciado del ejercicio?

**Respuesta:**

El **placebo pasa** en ambos: al permutar el tratamiento, el confusor omitido deja de importar y el efecto estimado vuelve a ≈ 0. Lo que sí delata el sabotaje es el **balance** (la covariable quitada queda desbalanceada tras ponderar) y el **efecto frente a la referencia congelada** (se mueve varias veces su error estándar). La afirmación «quitar un control rompe el placebo» es incorrecta; la puerta necesita las otras pruebas.



## 8. El script `make ci`: código de salida y bloqueo del registro (Completa tú)
`ejecutar_ci(etapa_sw, etapa_est, registrar)` implementa el Listado 20.1 con códigos de salida:
- Falla una prueba de **software** → devuelve **1** (no se corre la etapa lenta; no se registra).
- Software ok pero falla una prueba **estadística** → devuelve **2** («revisión humana»), **no registra**.
- Todo pasa → llama a `registrar()` y devuelve **0**.

In [ ]:
# ✏️ Completa tú — ejecutar_ci
# Pista: Recorre primero la etapa de software; solo si pasa, evalúa la estadística; llama a registrar() al final.
def ejecutar_ci(etapa_sw, etapa_est, registrar):
    """etapa_*: dict nombre -> función() -> bool. Devuelve 0, 1 (software) o 2 (estadística) y registra SOLO si devuelve 0."""
    raise NotImplementedError("completa ejecutar_ci")

In [ ]:
def suite_ci(ejecutar):
    reg = []; ok = lambda: True; no = lambda: False
    a = ejecutar({"s": ok}, {"e": ok}, lambda: reg.append(1)); n_a = len(reg)
    b = ejecutar({"s": no}, {"e": ok}, lambda: reg.append(1)); c = ejecutar({"s": ok}, {"e": no}, lambda: reg.append(1))
    return (a == 0 and n_a == 1, b == 1, c == 2, len(reg) == 1)
print("suite de la puerta:", suite_ci(ejecutar_ci))
assert all(suite_ci(ejecutar_ci)), "tu puerta no cumple el contrato (0 / 1 / 2 y registra solo con 0)"
# --- Prueba de mutación: ¿tu suite detecta una puerta que siempre devuelve 0?
puerta_mutante = lambda sw, est, reg: 0
print("¿la suite detecta la puerta que siempre devuelve 0?", "sí ✔" if not all(suite_ci(puerta_mutante)) else "NO ✘ (endurece la suite)")

In [ ]:
REGISTRO = Path(tempfile.mkdtemp(prefix="registro_"))
def registrar_bundle(): (REGISTRO / "dml_estimator.json").write_text('{"theta": %f}' % TH_REF)
ETAPA_EST = lambda covs: etapa_estadistica(REF, covs)
c_ok = ejecutar_ci(ETAPA_SW, ETAPA_EST(COVS), registrar_bundle)
print("modelo sano → código", c_ok, "· archivos registrados:", [p.name for p in REGISTRO.iterdir()])
shutil.rmtree(REGISTRO); REGISTRO.mkdir()
c_mal = ejecutar_ci(ETAPA_SW, ETAPA_EST(COVS_MAL), registrar_bundle)
print(f"modelo sin {CLAVE} → código", c_mal, "· archivos registrados:", [p.name for p in REGISTRO.iterdir()])

## 9. Señal de alerta: registrar en el mismo paso que las pruebas de software
El antipatrón: registrar apenas pasa la etapa 1. Comprobamos qué pasa con el modelo sabotado.

In [ ]:
def ci_mal(etapa_sw, etapa_est, registrar):
    if all(f() for f in etapa_sw.values()):
        registrar(); return 0                                     # ← nunca consulta la etapa estadística
    return 1
shutil.rmtree(REGISTRO); REGISTRO.mkdir()
print(f"antipatrón con el modelo sin {CLAVE} → código", ci_mal(ETAPA_SW, ETAPA_EST(COVS_MAL), registrar_bundle), "· registrado:", bool(list(REGISTRO.iterdir())), "← un modelo sesgado llegó al registro")

## 10. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Bloquea la puerta local el registro de un modelo que perdió un control clave?",
    metodo="CI en dos etapas: software rápida y estadística (hash de referencia, placebo con refit, balance, efecto vs referencia); código de salida 2 = revisión humana",
    estimando="ATE de A2 frente a la referencia congelada",
    numero_ic=fmt_ci(TH_REF, SE_REF) + f"; sin {CLAVE}: {th_m:.3f}",
    supuesto=supuesto("observable", "medido", f"el sabotaje (quitar {CLAVE}) da código {c_mal} y no se registra; el placebo, solo, no lo detecta"),
    decision="alertar" if c_mal == 2 else "continuar", lab="L18")

---
## Qué queda para el libro
Quedan en el libro: `pytest`, Docker y `compileall` del Listado 20.1 (aquí se simulan con funciones), MLflow/DVC/Great Expectations y el ejercicio conceptual del test DML-IV. El CI remoto queda fuera del flujo principal; el curso usa Make y pruebas locales (P-01).

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*